# Project Master Runner

This is the central entry point for the **Ecommerce Sentiment Analysis** project. It imports the logic from internal notebooks and executes the analysis pipeline.

In [1]:
# Load the DataExploration class from the notebooks folder
%run notebooks/DataExploration.ipynb

# Define the dataset path
data_path = 'Ecommerce_dataset/train_data.csv'

# Instantiate and run the analysis
eda = DataExploration(data_path)
eda.get_summary()
eda.get_sentiment_distribution()
eda.remove_nulls(columns=['reviews.text', 'sentiment'])



Dataset loaded successfully with 4000 rows and 8 columns.

DATASET SUMMARY

--- First 5 rows ---
                                                name   brand  \
0  All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi...  Amazon   
1        Amazon - Echo Plus w/ Built-In Hub - Silver  Amazon   
2  Amazon Echo Show Alexa-enabled Bluetooth Speak...  Amazon   
3  Fire HD 10 Tablet, 10.1 HD Display, Wi-Fi, 16 ...  Amazon   
4  Brand New Amazon Kindle Fire 16gb 7" Ips Displ...  Amazon   

                                          categories  \
0  Electronics,iPad & Tablets,All Tablets,Fire Ta...   
1  Amazon Echo,Smart Home,Networking,Home & Tools...   
2  Amazon Echo,Virtual Assistant Speakers,Electro...   
3  eBook Readers,Fire Tablets,Electronics Feature...   
4  Computers/Tablets & Networking,Tablets & eBook...   

             primaryCategories              reviews.date  \
0                  Electronics  2016-12-26T00:00:00.000Z   
1         Electronics,Hardware  2018-01-17T00:00:00.000Z   
2

#### So the data have no null values

In [2]:
#Check the split of the training data
positive = (eda.df["sentiment"] == "Positive").sum()
neutral = (eda.df["sentiment"] == "Neutral").sum()
negative = (eda.df["sentiment"] == "Negative").sum()

print(f"Positive: {positive}")
print(f"Neutral: {neutral}")
print(f"Negative: {negative}")


Positive: 3749
Neutral: 158
Negative: 93


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# ── Hardware Setup ────────────────────────────────────────────────────────
tf.config.threading.set_inter_op_parallelism_threads(0)
tf.config.threading.set_intra_op_parallelism_threads(0)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs available:", gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# ── Data Preparation ──────────────────────────────────────────────────────
encoder = LabelEncoder()
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
eda.df["sentiment"] = encoder.fit_transform(eda.df["sentiment"])
# LabelEncoder sorts alphabetically: Negative=0, Neutral=1, Positive=2
print("Label mapping:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

reviews = eda.df["reviews.text"].tolist()
labels  = eda.df["sentiment"].tolist()

# ── Class Weights ─────────────────────────────────────────────────────────
# Penalise misclassification of minority classes (Negative/Neutral) more heavily
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)
class_weight_dict = dict(enumerate(class_weights_array))
print("Class weights:", {encoder.classes_[k]: round(v, 2) for k, v in class_weight_dict.items()})
# Expected approx ➜  Negative: 14.3,  Neutral: 8.4,  Positive: 0.36

# ── Tokenisation ──────────────────────────────────────────────────────────
encodedInput = tokenizer(
    reviews,
    max_length=248,
    padding="max_length",
    truncation=True,
    return_tensors="tf"
)
input_ids      = encodedInput["input_ids"]
attention_mask = encodedInput["attention_mask"]
labels_tensor  = tf.cast(tf.convert_to_tensor(labels), tf.int32)

# ── tf.data Pipeline ──────────────────────────────────────────────────────
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

dataset = tf.data.Dataset.from_tensor_slices(
    ((input_ids, attention_mask), labels_tensor)
)
dataset = (dataset
    .shuffle(buffer_size=1000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ── Model Definition ───────────────────────────────────────────────────────
with tf.device("/GPU:0"):
    bert = TFBertModel.from_pretrained("bert-base-uncased", use_safetensors=False)

    input_ids_in      = tf.keras.Input(shape=(248,), dtype=tf.int32, name="input_ids")
    attention_mask_in = tf.keras.Input(shape=(248,), dtype=tf.int32, name="attention_mask")

    bert_output = bert({"input_ids": input_ids_in, "attention_mask": attention_mask_in})
    cls_vector  = bert_output.pooler_output

    x      = tf.keras.layers.Dense(64, activation="relu")(cls_vector)
    x      = tf.keras.layers.Dropout(0.2)(x)
    output = tf.keras.layers.Dense(3, activation="softmax")(x)

    model = tf.keras.Model(inputs=[input_ids_in, attention_mask_in], outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

model.summary()

# ── Training with Class Weights ────────────────────────────────────────────
model.fit(
    dataset,
    epochs=3,
    class_weight=class_weight_dict   # ← forces model to learn minority classes
)

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Label mapping: {0: 0, 1: 1, 2: 2}
Class weights: {0: 14.34, 1: 8.44, 2: 0.36}


Some layers from the model checkpoint at bert-base-uncased were not used when initializing TFBertModel: ['mlm___cls', 'nsp___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertModel were initialized from the model checkpoint at bert-base-uncased.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 attention_mask (InputLayer  [(None, 248)]                0         []                            
 )                                                                                                
                                                                                                  
 input_ids (InputLayer)      [(None, 248)]                0         []                            
                                                                                                  
 tf_bert_model_2 (TFBertMod  TFBaseModelOutputWithPooli   1094822   ['attention_mask[0][0]',      
 el)                         ngAndCrossAttentions(last_   40         'input_ids[0][0]']           
                             hidden_state=(None, 248, 7                                     

2026-03-28 00:23:16.723504: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node Adam/AssignAddVariableOp_10.


 18/125 [===>..........................] - ETA: 1:56 - loss: 1.4138 - accuracy: 0.2778